In [5]:
import pandas as pd
cols = ["ResponseId","Age","EdLevel","Employment","RemoteWork","DevType","OrgSize",
        "YearsCode","WorkExp","Country","AISelect","AIAcc","JobSat","ConvertedCompYearly"]
df = pd.read_csv("data/survey_results_public.csv", usecols=cols)
print(df.shape, df.duplicated("ResponseId").sum())
print(df.isna().mean().round(3))
print(df["YearsCode"].unique()[:15])
print(df["ConvertedCompYearly"].describe())
print(df["JobSat"].value_counts(dropna=False).head(12))

(49191, 14) 0
ResponseId             0.000
Age                    0.000
EdLevel                0.021
Employment             0.017
WorkExp                0.128
YearsCode              0.125
DevType                0.112
OrgSize                0.305
RemoteWork             0.313
Country                0.280
AISelect               0.315
AIAcc                  0.323
ConvertedCompYearly    0.513
JobSat                 0.458
dtype: float64
[14. 10. 12.  5. 22. 20. 13. 30. 15.  9.  6. nan  3. 40.  8.]
count    2.394700e+04
mean     1.017615e+05
std      4.617569e+05
min      1.000000e+00
25%      3.817100e+04
50%      7.532000e+04
75%      1.205960e+05
max      5.000000e+07
Name: ConvertedCompYearly, dtype: float64
JobSat
NaN     22521
8.0      6972
7.0      5590
9.0      3598
6.0      3230
10.0     2924
5.0      1896
3.0       811
4.0       803
2.0       439
0.0       242
1.0       165
Name: count, dtype: int64


In [6]:
import numpy as np

# 1. Check Age values first
print(df["Age"].value_counts(dropna=False))

# 2. Age: 'Prefer not to say' -> missing
df["Age"] = df["Age"].replace("Prefer not to say", np.nan)

# 3. Logic check: work experience can't exceed years coding
df["flag_exp_logic"] = df["WorkExp"] > df["YearsCode"]
print("WorkExp > YearsCode:", df["flag_exp_logic"].sum())

# 4. Logic check: very young + very experienced
young = df["Age"].isin(["Under 18 years old", "18-24 years old"])
df["flag_age_exp"] = young & (df["WorkExp"] > 10)
print("Young but 10+ yrs exp:", df["flag_age_exp"].sum())

# 5. Salary: valid range 1,000 to 1,000,000 USD, flag the rest (don't delete)
df["flag_comp_outlier"] = (df["ConvertedCompYearly"] < 1000) | (df["ConvertedCompYearly"] > 1_000_000)
df["comp_clean"] = df["ConvertedCompYearly"].where(~df["flag_comp_outlier"])
print("Salary outliers:", df["flag_comp_outlier"].sum())

# 6. Tidy text
df["Country"] = df["Country"].str.strip()

# 7. Save
df.to_csv("outputs/survey_clean.csv", index=False)

Age
25-34 years old      16519
35-44 years old      13241
18-24 years old       9210
45-54 years old       6275
55-64 years old       2626
65 years or older      942
Prefer not to say      378
Name: count, dtype: int64
WorkExp > YearsCode: 3564
Young but 10+ yrs exp: 48
Salary outliers: 770


Question 1: does AI usage differ by age group?

In [7]:
import pandas as pd, numpy as np
from scipy.stats import chi2_contingency, mannwhitneyu

df = pd.read_csv("outputs/survey_clean.csv")
print(df["AISelect"].value_counts(dropna=False))
print(df["RemoteWork"].value_counts(dropna=False))

d = df.dropna(subset=["Age", "AISelect"])
ct = pd.crosstab(d["Age"], d["AISelect"])
pct = pd.crosstab(d["Age"], d["AISelect"], normalize="index").mul(100).round(1)

chi2, p, dof, _ = chi2_contingency(ct)
v = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
print(pct)
print(f"chi2={chi2:.1f}, p={p:.4g}, Cramer's V={v:.3f}")
pct.to_csv("outputs/crosstab_age_ai.csv")

AISelect
Yes, I use AI tools daily                      15883
NaN                                            15471
Yes, I use AI tools weekly                      5958
No, and I don't plan to                         5454
Yes, I use AI tools monthly or infrequently     4628
No, but I plan to soon                          1797
Name: count, dtype: int64
RemoteWork
NaN                                                                             15411
Remote                                                                          10931
Hybrid (some remote, leans heavy to in-person)                                   6732
In-person                                                                        6042
Hybrid (some in-person, leans heavy to flexibility)                              5831
Your choice (very flexible, you can come in when you want or just as needed)     4244
Name: count, dtype: int64
AISelect           No, and I don't plan to  No, but I plan to soon  \
Age                     

Question 3: does job satisfaction differ by work mode?

In [9]:
d2 = df.dropna(subset=["RemoteWork", "JobSat"])
a = d2.loc[d2["RemoteWork"] == "Remote", "JobSat"]
b = d2.loc[d2["RemoteWork"] == "In-person", "JobSat"]
u, p2 = mannwhitneyu(a, b)
print(len(a), len(b), a.median(), b.median(), f"p={p2:.4g}")

7908 3496 8.0 7.0 p=1.866e-18


In [10]:
# effect size for the Mann-Whitney result
r = 1 - 2*u/(len(a)*len(b))
print("rank-biserial r =", round(r, 3))   # sign shows direction, |r|<0.1 negligible

# --- raking ---
d = df.dropna(subset=["Age", "RemoteWork", "AISelect"]).copy()
d = d[d["Age"] != "Prefer not to say"]
d["Work"] = d["RemoteWork"].map(lambda x: "Remote" if x == "Remote"
            else "In-person" if x == "In-person" else "Hybrid/Flexible")

targets = {
 "Age":  {"18-24 years old":.20,"25-34 years old":.30,"35-44 years old":.25,
          "45-54 years old":.15,"55-64 years old":.07,"65 years or older":.03},
 "Work": {"Remote":.30,"Hybrid/Flexible":.45,"In-person":.25},
}

def rake(d, targets, iters=100, tol=1e-6):
    w = np.ones(len(d))
    for _ in range(iters):
        old = w.copy()
        for var, tgt in targets.items():
            share = pd.Series(w, index=d.index).groupby(d[var]).sum() / w.sum()
            factor = d[var].map({k: tgt[k] / share[k] for k in tgt})
            w = w * factor.values
        if np.abs(w - old).max() < tol: break
    return w

d["w"] = rake(d, targets)
d["w"] = d["w"].clip(upper=5 * d["w"].mean())   # trim extreme weights

# check: weighted margins should match targets
for var in targets:
    print((d.groupby(var)["w"].sum() / d["w"].sum()).round(3))

n_eff = d["w"].sum()**2 / (d["w"]**2).sum()
print("n =", len(d), "| effective n =", round(n_eff))

# does weighting change the headline?
daily = d["AISelect"].eq("Yes, I use AI tools daily")
print("Unweighted daily AI:", round(daily.mean()*100, 1))
print("Weighted daily AI:  ", round((daily*d["w"]).sum()/d["w"].sum()*100, 1))
d.to_csv("outputs/survey_weighted.csv", index=False)

rank-biserial r = -0.101
Age
18-24 years old      0.20
25-34 years old      0.30
35-44 years old      0.25
45-54 years old      0.15
55-64 years old      0.07
65 years or older    0.03
Name: w, dtype: float64
Work
Hybrid/Flexible    0.45
In-person          0.25
Remote             0.30
Name: w, dtype: float64
n = 26497 | effective n = 23345
Unweighted daily AI: 48.4
Weighted daily AI:   47.8
